# Pixels to Predictions — Full Pipeline (Target: 0.90+)

## Architecture
1. **Stage A**: Self-caption every image using base SmolVLM
2. **Stage B**: Train diverse models with caption-augmented prompts + solution field
3. **Stage C**: Evaluate with TTA + ensemble
4. **Stage D**: Generate submission

**Current best:** 0.768 LB | **Target:** 0.90+

**GPU:** A100 (40GB) | **Estimated time:** 4-6 hours total

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Mount Drive & Setup
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/kaggle_final_competition")
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
CAPTION_DIR = PROJECT_ROOT / "captions"

for d in [CHECKPOINT_DIR, SUBMISSION_DIR, CAPTION_DIR]:
    d.mkdir(exist_ok=True)

print("Data dir:", sorted(os.listdir(DATA_DIR)))
print("Checkpoints:", sorted(os.listdir(CHECKPOINT_DIR)))

Mounted at /content/drive
Data dir: ['images', 'sample_submission.csv', 'test.csv', 'train.csv', 'val.csv']
Checkpoints: ['lora_r16_lr0.0002_sanity_ep1_best', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch12', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch24', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch36', 'lora_r16_lr0.0002_sanity_ep1_epoch1_batch48', 'lora_r16_lr0.0002_sanity_ep1_epoch1_final', 'lora_r16_lr0.0002_sanity_ep1_training_log.json', 'lora_r16_lr0.0002_sanity_ep1_val_results.json', 'phase0_val_predictions.csv', 'phase0_zero_shot_results.json', 'v2_seed42_r16_lr3e-05_ep2_ep1_b194', 'v2_seed42_r16_lr3e-05_ep2_ep1_b388', 'v2_seed42_r16_lr3e-05_ep2_ep1_b582', 'v2_seed42_r16_lr3e-05_ep2_ep1_b776', 'v2_seed42_r16_lr3e-05_ep2_ep2_b194', 'v2_seed42_r16_lr3e-05_ep2_ep2_b388', 'v2_seed42_r16_lr3e-05_ep2_ep2_b582', 'v2_seed42_r16_lr3e-05_ep2_ep2_b776', 'v2_seed42_r16_lr3e-05_ep2_final', 'v2_seed42_r16_lr3e-05_ep2_log.json', 'v2_seed42_r16_lr3e-05_ep2_val_results.json']


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: Install Packages
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.5 MB/s eta 0:00:00


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: Imports & Master Config
# ══════════════════════════════════════════════════════════════════════════════
import json
import random
import time
import gc
import copy
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: Load Data
# ══════════════════════════════════════════════════════════════════════════════
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Train has 'solution' column: {'solution' in train_df.columns}")
print(f"Solutions available: {train_df['solution'].notna().sum()}/{len(train_df)}")

Train: 3109 | Val: 1048 | Test: 1008
Train has 'solution' column: True
Solutions available: 2580/3109


---
## STAGE A: Self-Captioning

Use base SmolVLM to describe every image. This converts visual info → text,
playing to the model's language strength. The VCASFT paper shows this gives
significant accuracy boosts on ScienceQA.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: Load Base Model for Captioning
# ══════════════════════════════════════════════════════════════════════════════
from transformers import AutoProcessor, AutoModelForVision2Seq

print("Loading base model for captioning...")
cap_processor = AutoProcessor.from_pretrained(MODEL_ID)
if cap_processor.tokenizer.pad_token is None:
    cap_processor.tokenizer.pad_token = cap_processor.tokenizer.eos_token

cap_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
cap_model.eval()
print(f"Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading base model for captioning...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model loaded. VRAM: 1.0 GB


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: Caption Generation Function
# ══════════════════════════════════════════════════════════════════════════════

def generate_captions(df, model, processor, data_dir, batch_size=4, max_new_tokens=100):
    """
    Generate image captions using the base SmolVLM model.
    Uses batched generation for speed.
    """
    captions = {}
    caption_prompt = "<image>\nDescribe this image in detail. Focus on any diagrams, charts, maps, labels, numbers, arrows, or scientific content visible."

    pbar = tqdm(range(0, len(df), batch_size), desc="Captioning", leave=True)

    for start in pbar:
        end = min(start + batch_size, len(df))
        chunk = df.iloc[start:end]

        images = []
        prompts = []
        ids = []

        for _, row in chunk.iterrows():
            img = Image.open(data_dir / row["image_path"]).convert("RGB")
            images.append(img)
            prompts.append(caption_prompt)
            ids.append(row["id"])

        inputs = processor(
            text=prompts, images=images,
            return_tensors="pt", padding=True,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

        with torch.inference_mode():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
            )

        # Decode only the generated part (skip input tokens)
        input_len = inputs["input_ids"].shape[1]
        for i, sample_id in enumerate(ids):
            generated = processor.tokenizer.decode(
                gen_ids[i, input_len:], skip_special_tokens=True
            ).strip()
            captions[sample_id] = generated

        pbar.set_postfix({"done": len(captions)})

        # Periodic memory cleanup
        del inputs, gen_ids
        if (start // batch_size) % 20 == 0:
            torch.cuda.empty_cache()

    return captions

print("Caption function ready.")

Caption function ready.


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: Generate Captions (or load from checkpoint)
# ══════════════════════════════════════════════════════════════════════════════
caption_file = CAPTION_DIR / "all_captions.json"

if caption_file.exists():
    print("Loading cached captions...")
    with open(caption_file) as f:
        all_captions = json.load(f)
    print(f"Loaded {len(all_captions)} captions from cache.")
else:
    print("Generating captions for ALL splits...")
    all_captions = {}

    print("\n--- Train set ---")
    train_caps = generate_captions(train_df, cap_model, cap_processor, DATA_DIR, batch_size=8)
    all_captions.update(train_caps)

    # Checkpoint after train
    with open(caption_file, "w") as f:
        json.dump(all_captions, f)
    print(f"Checkpoint: {len(all_captions)} captions saved.")

    print("\n--- Val set ---")
    val_caps = generate_captions(val_df, cap_model, cap_processor, DATA_DIR, batch_size=8)
    all_captions.update(val_caps)

    print("\n--- Test set ---")
    test_caps = generate_captions(test_df, cap_model, cap_processor, DATA_DIR, batch_size=8)
    all_captions.update(test_caps)

    # Final save
    with open(caption_file, "w") as f:
        json.dump(all_captions, f)
    print(f"\nDone! {len(all_captions)} total captions saved to {caption_file}")

# Show examples
print("\n--- Sample captions ---")
for sample_id in list(all_captions.keys())[:3]:
    print(f"\n{sample_id}: {all_captions[sample_id][:200]}...")

Loading cached captions...
Loaded 5165 captions from cache.

--- Sample captions ---

train_07667: The image is a photograph of two frogs on a green surface. The frogs are positioned in such a way that their front legs are on top of each other, and their back legs are on top of each other. The frog...

train_02628: Answer questions based on the image.
Answer: No.semination

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#

#...

train_00927: The image is a photograph of a group of lions in a grassy field. The background is a blurred mix of green and brown, suggesting a natural setting. The foreground is dominated by the animals, which are...


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Free captioning model memory
# ══════════════════════════════════════════════════════════════════════════════
del cap_model, cap_processor
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")

VRAM after cleanup: 0.0 GB


---
## STAGE B: Training with Caption-Augmented Prompts

We train diverse models with:
- Image captions as additional context
- Solution field for richer supervision
- Different configs for ensemble diversity

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Caption-Augmented Prompt Builder
# ══════════════════════════════════════════════════════════════════════════════

def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def build_prompt_with_caption(row, captions, include_answer=False, use_solution=False):
    """
    Enhanced prompt with self-generated image caption + optional solution.
    """
    choices = row["choices"]
    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    caption = captions.get(str(row["id"]), "")

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"

    # Add caption as "Image description"
    if caption:
        prompt += f"Image description: {caption}\n\n"

    if lecture:
        prompt += f"Lecture: {lecture}\n\n"
    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        letter = CHOICE_LETTERS[ans]
        answer_text = f" {letter}. {choices[ans]}"

        # Optionally add solution/explanation
        if use_solution:
            solution = clean_text(row.get("solution", ""))
            if solution:
                answer_text += f"\nExplanation: {solution[:300]}"

        prompt += answer_text

    return prompt

def build_prompt_v2_with_caption(row, captions, include_answer=False):
    """v2 style but with caption. No solution field."""
    return build_prompt_with_caption(row, captions, include_answer, use_solution=False)

def build_prompt_v2_no_caption(row, captions, include_answer=False):
    """Original v2 style, no caption (for diversity)."""
    choices = row["choices"]
    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"

    if include_answer:
        ans = int(row["answer"])
        letter = CHOICE_LETTERS[ans]
        prompt += f" {letter}. {choices[ans]}"

    return prompt

# Test
print("=== With caption + solution ===")
sample = train_df.iloc[0]
p = build_prompt_with_caption(sample, all_captions, include_answer=True, use_solution=True)
print(p[:500])
print("...")
print(p[-200:])

=== With caption + solution ===
<image>
You are solving a science multiple-choice question.
Use the image and text to choose the best answer.
Answer with ONLY the letter and the full text of the correct choice.

Image description: The image is a photograph of two frogs on a green surface. The frogs are positioned in such a way that their front legs are on top of each other, and their back legs are on top of each other. The frog on the left is larger and has a more pronounced black stripe along its back, while the frog on the r
...
f water. Use this information to determine why this behavior can increase the reproductive success of a male Amazonian poison frog.
Choice "Amazonian poison frogs live in tropical forests in northern 


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10: Training Dataset + Collator
# ══════════════════════════════════════════════════════════════════════════════

class VQATrainDataset(Dataset):
    def __init__(self, df, processor, data_dir, captions, prompt_fn, img_size=224):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.captions = captions
        self.prompt_fn = prompt_fn
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        image = image.resize((self.img_size, self.img_size), Image.BICUBIC)

        prompt_text = self.prompt_fn(row, self.captions, include_answer=False)
        full_text = self.prompt_fn(row, self.captions, include_answer=True)

        full_inputs = self.processor(
            text=[full_text], images=[image],
            return_tensors="pt", truncation=False
        )
        prompt_inputs = self.processor(
            text=[prompt_text], images=[image],
            return_tensors="pt", truncation=False
        )

        return {
            "input_ids": full_inputs["input_ids"].squeeze(0),
            "attention_mask": full_inputs["attention_mask"].squeeze(0),
            "pixel_values": full_inputs["pixel_values"].squeeze(0),
            "prompt_len": prompt_inputs["input_ids"].shape[1],
        }


def collate_train(batch, pad_id):
    """Left-pad, mask everything except answer tokens."""
    max_len = max(x["input_ids"].shape[0] for x in batch)

    input_ids_list, attention_mask_list, labels_list, pixel_values_list = [], [], [], []

    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len
        prompt_len = x["prompt_len"]

        input_ids = torch.cat([
            torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype),
            x["input_ids"],
        ])
        attention_mask = torch.cat([
            torch.zeros(pad_len, dtype=x["attention_mask"].dtype),
            x["attention_mask"],
        ])

        labels = torch.full_like(input_ids, -100)
        answer_start = pad_len + prompt_len
        labels[answer_start:] = input_ids[answer_start:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
        pixel_values_list.append(x["pixel_values"])

    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list),
        "pixel_values": torch.stack(pixel_values_list),
    }

print("Dataset + Collator ready.")

Dataset + Collator ready.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11: Model Loading + Training Function
# ══════════════════════════════════════════════════════════════════════════════
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel

def load_fresh_model(seed, lora_r=16, lora_alpha=32, lora_targets=None):
    """Load fresh base model with QLoRA."""
    if lora_targets is None:
        lora_targets = ["q_proj", "k_proj", "v_proj", "o_proj"]

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=lora_targets,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable: {trainable:,} ({trainable/1e6:.2f}M) | "
          f"{'OK' if trainable <= 5_000_000 else 'OVER LIMIT!'}")

    return model, processor


def train_one_model(model, processor, train_df, captions, prompt_fn,
                    exp_name, seed=42, epochs=2, lr=3e-5, batch_size=4):
    """Train and checkpoint one model. Returns model."""
    print(f"\n{'='*60}")
    print(f"TRAINING: {exp_name}")
    print(f"{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pad_id = processor.tokenizer.pad_token_id
    dataset = VQATrainDataset(train_df, processor, DATA_DIR, captions, prompt_fn, img_size=224)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        collate_fn=lambda b: collate_train(b, pad_id),
        num_workers=2, pin_memory=True,
    )

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01,
    )
    total_steps = len(loader) * epochs
    warmup_steps = int(total_steps * 0.05)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    print(f"Batches/epoch: {len(loader)} | Total steps: {total_steps}")

    model.train()
    log = []
    start_time = time.time()

    for epoch in range(epochs):
        epoch_losses = []
        pbar = tqdm(enumerate(loader), total=len(loader),
                    desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for batch_idx, batch in pbar:
            batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            epoch_losses.append(loss.item())
            avg_loss = np.mean(epoch_losses[-20:])
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "avg": f"{avg_loss:.4f}",
                              "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

            if (batch_idx + 1) % 50 == 0:
                log.append({"epoch": epoch+1, "batch": batch_idx+1,
                            "loss": loss.item(), "avg": avg_loss})

            # Mid-epoch checkpoint
            ckpt_interval = max(len(loader) // 4, 1)
            if (batch_idx + 1) % ckpt_interval == 0 and (batch_idx + 1) < len(loader):
                mid_path = CHECKPOINT_DIR / f"{exp_name}_ep{epoch+1}_b{batch_idx+1}"
                model.save_pretrained(str(mid_path))
                print(f"\n  Checkpoint: {mid_path.name}")

        print(f"\nEpoch {epoch+1} — Avg loss: {np.mean(epoch_losses):.4f}")

    elapsed = time.time() - start_time
    final_path = CHECKPOINT_DIR / f"{exp_name}_final"
    model.save_pretrained(str(final_path))

    with open(CHECKPOINT_DIR / f"{exp_name}_log.json", "w") as f:
        json.dump(log, f, indent=2)

    print(f"Model saved: {final_path.name} ({elapsed/60:.1f} min)")

    # Cleanup training memory
    del optimizer, scheduler, loader, dataset
    gc.collect()
    torch.cuda.empty_cache()

    return model

print("Training function ready.")

Training function ready.


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12: Inference Functions (Fixed Padding + Multi-form + TTA)
# ══════════════════════════════════════════════════════════════════════════════

def get_candidate_token_ids(tokenizer):
    """Get token IDs for each choice letter. Only unique IDs per letter."""
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " " + letter]  # No period forms to avoid shared tokens
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc:
                ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids


def predict_batch(model, processor, df_batch, captions, prompt_fn,
                  candidate_ids, use_native=True):
    """Score batch with FIXED padding position lookup."""
    images = []
    for _, row in df_batch.iterrows():
        img = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
        if not use_native:
            img = img.resize((224, 224), Image.BICUBIC)
        images.append(img)

    prompts = [prompt_fn(row, captions, include_answer=False)
               for _, row in df_batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits

    # FIXED: use actual last non-pad position
    seq_lengths = inputs["attention_mask"].sum(dim=1)

    preds = []
    scores_all = []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        last_pos = int(seq_lengths[i].item()) - 1
        last_logits = logits[i, last_pos, :]
        log_probs = torch.log_softmax(last_logits, dim=-1)

        scores = []
        for ci in range(len(row["choices"])):
            tids = candidate_ids[CHOICE_LETTERS[ci]]
            scores.append(max(log_probs[tid].item() for tid in tids))
        preds.append(int(np.argmax(scores)))
        scores_all.append(scores)

    return preds, scores_all


def predict_batch_tta(model, processor, df_batch, captions, prompt_fn,
                      candidate_ids, K=4, use_native=True, rng=None):
    """
    Test-Time Augmentation: shuffle choice order K times, average scores.
    Only apply TTA to 3+ choice questions.
    """
    if rng is None:
        rng = random.Random(42)

    n_samples = len(df_batch)
    max_choices = max(len(row["choices"]) for _, row in df_batch.iterrows())
    accum = np.full((n_samples, max_choices), -1e9)

    for k in range(K):
        # Generate permutations
        perms = []
        for _, row in df_batch.iterrows():
            n = len(row["choices"])
            if k == 0 or n <= 2:  # identity for first pass and 2-choice
                perms.append(list(range(n)))
            else:
                p = list(range(n))
                rng.shuffle(p)
                perms.append(p)

        # Build shuffled prompts
        shuffled_rows = []
        for (_, row), perm in zip(df_batch.iterrows(), perms):
            shuffled_row = row.copy()
            shuffled_row["choices"] = [row["choices"][j] for j in perm]
            shuffled_rows.append(shuffled_row)

        shuffled_df = pd.DataFrame(shuffled_rows)
        _, scores = predict_batch(model, processor, shuffled_df, captions,
                                  prompt_fn, candidate_ids, use_native)

        # Map shuffled scores back to original positions
        for i, (perm, sc) in enumerate(zip(perms, scores)):
            for shuf_pos, orig_idx in enumerate(perm):
                if accum[i, orig_idx] < -1e8:  # first write
                    accum[i, orig_idx] = sc[shuf_pos]
                else:
                    accum[i, orig_idx] += sc[shuf_pos]

    preds = [int(np.argmax(accum[i, :len(row["choices"])]))
             for i, (_, row) in enumerate(df_batch.iterrows())]
    return preds, accum


def evaluate_full(model, processor, df, captions, prompt_fn,
                  candidate_ids, batch_size=8, use_native=True,
                  use_tta=False, tta_k=4, desc="Eval"):
    """Full evaluation with progress bar and running accuracy."""
    model.eval()
    all_preds = []
    all_scores = []
    correct = 0
    total = 0
    rng = random.Random(42)

    pbar = tqdm(range(0, len(df), batch_size), desc=desc, leave=True)

    for start in pbar:
        end = min(start + batch_size, len(df))
        batch_df = df.iloc[start:end]

        if use_tta:
            preds, scores = predict_batch_tta(
                model, processor, batch_df, captions, prompt_fn,
                candidate_ids, K=tta_k, use_native=use_native, rng=rng)
        else:
            preds, scores = predict_batch(
                model, processor, batch_df, captions, prompt_fn,
                candidate_ids, use_native=use_native)

        all_preds.extend(preds)
        all_scores.extend(scores if not use_tta else scores.tolist())

        if "answer" in df.columns:
            for p, (_, row) in zip(preds, batch_df.iterrows()):
                total += 1
                if p == int(row["answer"]):
                    correct += 1
            pbar.set_postfix({"acc": f"{correct/total:.4f} ({correct}/{total})"})

        torch.cuda.empty_cache()

    acc = correct / total if total > 0 else None
    return all_preds, all_scores, acc

print("Inference functions ready (with TTA + fixed padding).")

Inference functions ready (with TTA + fixed padding).


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13: Define Ensemble Configs
# ══════════════════════════════════════════════════════════════════════════════

# Each config is a different model for ensemble diversity
ENSEMBLE_CONFIGS = [
    {
        "name": "captionV2_r16_qkvo_lr3e5_ep2_seed42",
        "seed": 42,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 3e-5,
        "epochs": 2,
        "prompt_fn": build_prompt_v2_with_caption,  # WITH caption
    },
    {
        "name": "captionSol_r16_qkvo_lr3e5_ep2_seed123",
        "seed": 123,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 3e-5,
        "epochs": 2,
        "prompt_fn": lambda row, caps, inc: build_prompt_with_caption(
            row, caps, inc, use_solution=True),  # WITH caption + solution
    },
    {
        "name": "noCap_r16_qkvo_lr2e5_ep3_seed456",
        "seed": 456,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 2e-5,
        "epochs": 3,
        "prompt_fn": build_prompt_v2_no_caption,  # NO caption (diversity)
    },
    {
        "name": "captionV2_r16_qkvo_lr3e5_ep2_seed789",
        "seed": 789,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 3e-5,
        "epochs": 2,
        "prompt_fn": build_prompt_v2_with_caption,  # WITH caption, different seed
    },
    {
        "name": "captionV2_r16_qkvo_lr4e5_ep2_seed999",
        "seed": 999,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 4e-5,
        "epochs": 2,
        "prompt_fn": build_prompt_v2_with_caption,  # WITH caption, higher LR
    },
]

print(f"Defined {len(ENSEMBLE_CONFIGS)} ensemble configs:")
for cfg in ENSEMBLE_CONFIGS:
    print(f"  {cfg['name']}: r={cfg['lora_r']}, lr={cfg['lr']}, ep={cfg['epochs']}")

Defined 5 ensemble configs:
  captionV2_r16_qkvo_lr3e5_ep2_seed42: r=16, lr=3e-05, ep=2
  captionSol_r16_qkvo_lr3e5_ep2_seed123: r=16, lr=3e-05, ep=2
  noCap_r16_qkvo_lr2e5_ep3_seed456: r=16, lr=2e-05, ep=3
  captionV2_r16_qkvo_lr3e5_ep2_seed789: r=16, lr=3e-05, ep=2
  captionV2_r16_qkvo_lr4e5_ep2_seed999: r=16, lr=4e-05, ep=2


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 14: Train + Evaluate All Models (MAIN LOOP)
# ══════════════════════════════════════════════════════════════════════════════
# This is the big one. Each model: ~40 min train + ~10 min eval on A100.
# Total: ~4 hours for 5 models.

ensemble_results = []

for i, cfg in enumerate(ENSEMBLE_CONFIGS):
    print(f"\n\n{'#'*60}")
    print(f"# MODEL {i+1}/{len(ENSEMBLE_CONFIGS)}: {cfg['name']}")
    print(f"{'#'*60}")

    # Check if already trained
    final_path = CHECKPOINT_DIR / f"{cfg['name']}_final"
    results_path = CHECKPOINT_DIR / f"{cfg['name']}_results.json"

    if results_path.exists():
        print(f"Loading cached results from {results_path.name}")
        with open(results_path) as f:
            cached = json.load(f)
        ensemble_results.append(cached)
        continue

    # Load fresh model
    model, processor = load_fresh_model(
        seed=cfg["seed"],
        lora_r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_targets=cfg["lora_targets"],
    )

    # Train
    model = train_one_model(
        model, processor, train_df, all_captions,
        prompt_fn=cfg["prompt_fn"],
        exp_name=cfg["name"],
        seed=cfg["seed"],
        epochs=cfg["epochs"],
        lr=cfg["lr"],
        batch_size=4,
    )

    # Get candidate token IDs
    candidate_ids = get_candidate_token_ids(processor.tokenizer)

    # Evaluate on val (no TTA first for speed)
    print("\nEvaluating on val...")
    val_preds, val_scores, val_acc = evaluate_full(
        model, processor, val_df, all_captions,
        prompt_fn=cfg["prompt_fn"],
        candidate_ids=candidate_ids,
        batch_size=4, use_native=True,
        desc=f"Val ({cfg['name'][:20]})"
    )
    print(f"\nVal accuracy: {val_acc:.4f}")

    # Evaluate on val WITH TTA
    print("\nEvaluating on val with TTA (K=4)...")
    val_preds_tta, val_scores_tta, val_acc_tta = evaluate_full(
        model, processor, val_df, all_captions,
        prompt_fn=cfg["prompt_fn"],
        candidate_ids=candidate_ids,
        batch_size=4, use_native=True,
        use_tta=True, tta_k=4,
        desc=f"Val+TTA ({cfg['name'][:20]})"
    )
    print(f"Val accuracy (TTA): {val_acc_tta:.4f}")

    # Test inference (no TTA — we'll do TTA at ensemble level)
    print("\nRunning test inference...")
    test_preds, test_scores, _ = evaluate_full(
        model, processor, test_df, all_captions,
        prompt_fn=cfg["prompt_fn"],
        candidate_ids=candidate_ids,
        batch_size=4, use_native=True,
        desc=f"Test ({cfg['name'][:20]})"
    )

    # Save results
    result = {
        "name": cfg["name"],
        "val_acc": val_acc,
        "val_acc_tta": val_acc_tta,
        "val_preds": val_preds,
        "val_scores": val_scores,
        "test_preds": test_preds,
        "test_scores": test_scores,
        "config": {k: str(v) if callable(v) else v
                   for k, v in cfg.items() if k != "prompt_fn"},
    }
    ensemble_results.append(result)

    with open(results_path, "w") as f:
        json.dump(result, f)
    print(f"Results saved: {results_path.name}")

    # Save individual submission
    sub = pd.DataFrame({"id": test_df["id"], "answer": test_preds})
    sub.to_csv(SUBMISSION_DIR / f"submission_{cfg['name']}.csv", index=False)

    # Free model
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM freed: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print(f"\n\n{'='*60}")
print(f"ALL MODELS TRAINED AND EVALUATED")
print(f"{'='*60}")
for r in ensemble_results:
    print(f"  {r['name'][:40]}: val={r['val_acc']:.4f}, val_tta={r['val_acc_tta']:.4f}")



############################################################
# MODEL 1/5: captionV2_r16_qkvo_lr3e5_ep2_seed42
############################################################
Trainable: 4,161,536 (4.16M) | OK

TRAINING: captionV2_r16_qkvo_lr3e5_ep2_seed42
Batches/epoch: 778 | Total steps: 1556


Epoch 1/2:   0%|          | 0/778 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



  Checkpoint: captionV2_r16_qkvo_lr3e5_ep2_seed42_ep1_b194

  Checkpoint: captionV2_r16_qkvo_lr3e5_ep2_seed42_ep1_b388

  Checkpoint: captionV2_r16_qkvo_lr3e5_ep2_seed42_ep1_b582

  Checkpoint: captionV2_r16_qkvo_lr3e5_ep2_seed42_ep1_b776

Epoch 1 — Avg loss: 0.1304


Epoch 2/2:   0%|          | 0/778 [00:00<?, ?it/s]

...

Val accuracy (TTA): 0.7032

Running test inference...


Test (captionV2_r16_qkvo_l):   0%|          | 0/252 [00:00<?, ?it/s]

Results saved: captionV2_r16_qkvo_lr3e5_ep2_seed42_results.json
VRAM freed: 0.0 GB


############################################################
# MODEL 2/5: captionSol_r16_qkvo_lr3e5_ep2_seed123
############################################################
Trainable: 4,161,536 (4.16M) | OK

TRAINING: captionSol_r16_qkvo_lr3e5_ep2_seed123
Batches/epoch: 778 | Total steps: 1556


Epoch 1/2:   0%|          | 0/778 [00:00<?, ?it/s]

TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_1000/2827902105.py", line 22, in __getitem__
    prompt_text = self.prompt_fn(row, self.captions, include_answer=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: <lambda>() got an unexpected keyword argument 'include_answer'


---
## STAGE C: Ensemble

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 15: Ensemble Methods
# ══════════════════════════════════════════════════════════════════════════════

def ensemble_majority_vote(pred_lists):
    """Simple majority vote."""
    n = len(pred_lists[0])
    preds = []
    for i in range(n):
        votes = [pl[i] for pl in pred_lists]
        preds.append(Counter(votes).most_common(1)[0][0])
    return preds

def ensemble_score_average(score_lists, num_choices_list):
    """Average log-prob scores across models."""
    n = len(score_lists[0])
    preds = []
    for i in range(n):
        nc = num_choices_list[i]
        avg_scores = []
        for ci in range(nc):
            vals = []
            for sl in score_lists:
                s = sl[i]
                if isinstance(s, list) and ci < len(s):
                    vals.append(s[ci])
                elif isinstance(s, (np.ndarray,)) and ci < len(s):
                    vals.append(float(s[ci]))
            avg_scores.append(np.mean(vals) if vals else -1e9)
        preds.append(int(np.argmax(avg_scores)))
    return preds

def ensemble_weighted_vote(pred_lists, weights):
    """Weighted majority vote using val accuracy as weight."""
    n = len(pred_lists[0])
    preds = []
    for i in range(n):
        vote_scores = {}
        for pl, w in zip(pred_lists, weights):
            v = pl[i]
            vote_scores[v] = vote_scores.get(v, 0) + w
        preds.append(max(vote_scores, key=vote_scores.get))
    return preds

# Run all ensemble methods on val
val_actuals = [int(row["answer"]) for _, row in val_df.iterrows()]
val_num_choices = [len(row["choices"]) for _, row in val_df.iterrows()]
test_num_choices = [len(row["choices"]) for _, row in test_df.iterrows()]

val_pred_lists = [r["val_preds"] for r in ensemble_results]
val_score_lists = [r["val_scores"] for r in ensemble_results]
val_accs = [r["val_acc"] for r in ensemble_results]

# Majority vote
mv_preds = ensemble_majority_vote(val_pred_lists)
mv_acc = sum(p == a for p, a in zip(mv_preds, val_actuals)) / len(val_actuals)

# Score average
sa_preds = ensemble_score_average(val_score_lists, val_num_choices)
sa_acc = sum(p == a for p, a in zip(sa_preds, val_actuals)) / len(val_actuals)

# Weighted vote
wv_preds = ensemble_weighted_vote(val_pred_lists, val_accs)
wv_acc = sum(p == a for p, a in zip(wv_preds, val_actuals)) / len(val_actuals)

print(f"{'='*60}")
print(f"ENSEMBLE RESULTS (Validation)")
print(f"{'='*60}")
print(f"\nIndividual models:")
for r in ensemble_results:
    print(f"  {r['name'][:40]}: {r['val_acc']:.4f}")
print(f"\nEnsemble methods:")
print(f"  Majority vote:     {mv_acc:.4f}")
print(f"  Score average:     {sa_acc:.4f}")
print(f"  Weighted vote:     {wv_acc:.4f}")
print(f"\nBest individual:     {max(val_accs):.4f}")
best_ensemble_acc = max(mv_acc, sa_acc, wv_acc)
print(f"Best ensemble:       {best_ensemble_acc:.4f}")
print(f"Improvement:         {best_ensemble_acc - max(val_accs):+.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 16: Generate Final Submissions
# ══════════════════════════════════════════════════════════════════════════════

test_pred_lists = [r["test_preds"] for r in ensemble_results]
test_score_lists = [r["test_scores"] for r in ensemble_results]

# Apply best ensemble method to test
methods = {
    "majority_vote": (mv_acc, ensemble_majority_vote(test_pred_lists)),
    "score_average": (sa_acc, ensemble_score_average(test_score_lists, test_num_choices)),
    "weighted_vote": (wv_acc, ensemble_weighted_vote(test_pred_lists, val_accs)),
}

best_method_name = max(methods, key=lambda k: methods[k][0])
best_test_preds = methods[best_method_name][1]

# Save all submissions
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

for method_name, (val_a, test_p) in methods.items():
    sub = pd.DataFrame({"id": test_df["id"], "answer": test_p})
    assert len(sub) == len(sample_sub), f"Length mismatch for {method_name}"
    sub.to_csv(SUBMISSION_DIR / f"submission_{method_name}.csv", index=False)

# Best submission
best_sub = pd.DataFrame({"id": test_df["id"], "answer": best_test_preds})
best_sub.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

print(f"{'='*60}")
print(f"FINAL SUBMISSIONS")
print(f"{'='*60}")
for method_name, (val_a, test_p) in methods.items():
    marker = " <-- BEST" if method_name == best_method_name else ""
    print(f"  {method_name}: val={val_a:.4f}{marker}")
print(f"\nBest method: {best_method_name}")
print(f"\nPrediction distribution:")
print(best_sub["answer"].value_counts().sort_index())
print(f"\nFiles saved to: {SUBMISSION_DIR}")
print(f"\nUpload submission.csv to Kaggle!")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 17: Save Complete Results Summary
# ══════════════════════════════════════════════════════════════════════════════

summary = {
    "individual_models": [
        {"name": r["name"], "val_acc": r["val_acc"], "val_acc_tta": r["val_acc_tta"]}
        for r in ensemble_results
    ],
    "ensemble": {
        "majority_vote_val": mv_acc,
        "score_average_val": sa_acc,
        "weighted_vote_val": wv_acc,
        "best_method": best_method_name,
        "best_val_acc": best_ensemble_acc,
    },
    "baselines": {
        "zero_shot": 0.5563,
        "ta_baseline": 0.6781,
        "teammate_best_lb": 0.768,
        "top_team_lb": 0.899,
    },
    "techniques_used": [
        "Self-captioning (SmolVLM generates image descriptions)",
        "Solution field in training (chain-of-thought supervision)",
        "Answer-phrase training (not letter-only)",
        "Left-padded collator with answer-only loss masking",
        "Native image resolution at inference",
        "Multi-form token scoring (A, ' A')",
        "Fixed padding position for batched inference",
        "Test-time augmentation (choice shuffling K=4)",
        "5-model diverse ensemble (3 methods compared)",
    ],
}

with open(CHECKPOINT_DIR / "final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nSummary saved to checkpoints/final_summary.json")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 18: Download Best Submission
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import files

# Download the best ensemble submission
files.download(str(SUBMISSION_DIR / "submission.csv"))

# Also download all individual + ensemble submissions for comparison
for f_name in sorted(os.listdir(SUBMISSION_DIR)):
    if f_name.endswith(".csv"):
        print(f"Available: {f_name}")

# Auto-delete runtime to save credits
import os
os.kill(os.getpid(), 9)

---
## Run Order

1. **Cells 1-4:** Setup, install, imports, load data (~2 min)
2. **Cells 5-8:** STAGE A — Self-captioning (~30 min for all splits)
3. **Cells 9-12:** Define prompt builders, dataset, inference functions (~1 min)
4. **Cell 13:** Define ensemble configs (~1 min)
5. **Cell 14:** STAGE B — Train + evaluate ALL models (~4 hours)
   - Each model: ~40 min train + ~10 min eval
   - Checkpoints saved after each model
   - If Colab disconnects, re-run — cached results will be loaded
6. **Cell 15:** STAGE C — Ensemble comparison (~1 min)
7. **Cells 16-18:** STAGE D — Generate + download submissions (~1 min)

### Total estimated time: ~5 hours on A100

### Techniques stack:
- Self-captioning → converts weak vision to strong text
- Solution field → teaches reasoning patterns
- 5-model ensemble → corrects individual model errors
- TTA → free +1-2% from choice-order invariance
- Fixed padding → correct logit extraction
- Native inference → preserves image detail